# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a structured guide for loading and exploring a dataset using the `mlcroissant` library. All schema elements are referenced by their unique `@id` per dataset best practices.

### Dataset Source
The dataset Croissant schema URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure the mlcroissant library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load catalog (metadata) and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # Metadata is an object, not a dict
print(f"{getattr(metadata, 'name', '')}: {getattr(metadata, 'description', '')}\n")
print(f"Version: {getattr(metadata, 'version', '')}\nPublished: {getattr(metadata, 'datePublished', '')}")

## 2. Data Overview
Review available record sets, fields, and their IDs. This uses `@id` for all entities.

Let's find out which record sets are available in the dataset, and then enumerate their fields and columns (by `@id`).

In [ ]:
# Helper: print available record sets with their `@id` and fields
if hasattr(metadata, 'record_sets'):
    record_sets = metadata.record_sets
elif hasattr(metadata, 'recordSet'):
    record_sets = metadata.recordSet
else:
    record_sets = []

if not record_sets or len(record_sets) == 0:
    print("No record sets found in the metadata.\n")

record_set_ids = []
for rs in record_sets:
    rs_id = getattr(rs, '@id', None) or getattr(rs, 'id', None)
    if not rs_id:
        continue
    record_set_ids.append(rs_id)
    print(f"RecordSet @id: {rs_id}")
    print(f"  Name: {getattr(rs, 'name', '')}")
    # Fields
    fields = getattr(rs, 'fields', getattr(rs, 'field', []))
    field_ids = []
    if fields:
        for f in fields:
            f_id = getattr(f, '@id', None) or getattr(f, 'id', None)
            field_ids.append(f_id)
            print(f"    Field @id: {f_id:40} Name: {getattr(f, 'name', '')}")
    # Columns (if exist)
    columns = getattr(rs, 'columns', getattr(rs, 'column', []))
    if columns:
        for c in columns:
            c_id = getattr(c, '@id', None) or getattr(c, 'id', None)
            print(f"    Column @id: {c_id:37} Name: {getattr(c, 'name', '')}")
    print()

Below we will print some sample records from each record set, referencing the record set by its `@id`:

In [ ]:
# Show a few records from each record set (referenced by @id)
if not record_set_ids:
    print("No record sets available for data loading.")
else:
    for recset_id in record_set_ids:
        print(f"\n==== Records for RecordSet @id: {recset_id} ====")
        try:
            for idx, rec in enumerate(dataset.records(record_set=recset_id)):
                print(json.dumps(rec, indent=2))
                if idx >= 1:  # Show only first two records
                    break
        except Exception as e:
            print(f"  Error loading records: {e}")

## 3. Data Extraction
Load data from all available record sets into pandas DataFrames for analysis. All references use the record set and field `@id` values.

In [ ]:
dataframes = {}

for recset_id in record_set_ids:
    try:
        print(f"Loading records for RecordSet @id: {recset_id}")
        recs = list(dataset.records(record_set=recset_id))
        dataframes[recset_id] = pd.DataFrame(recs)
        print(f"  Columns: {dataframes[recset_id].columns.tolist()}")
    except Exception as ex:
        print(f"  Could not load: {ex}")

# If only one recordset or if you want to select a particular record set, use it below
if record_set_ids:
    main_recset_id = record_set_ids[0]
    print(f"\nFirst DataFrame columns for RecordSet @id {main_recset_id}:")
    display_cols = dataframes[main_recset_id].columns.tolist()
    print(display_cols)
    dataframes[main_recset_id].head()

## 4. Exploratory Data Analysis (EDA)
We'll illustrate data processing steps: filtering records on a numeric field, normalizing, and grouping, using `@id` references for fields. Update the field names or `@id` as needed after inspecting actual columns.

In [ ]:
# Select a record set and numeric field to analyze (by @id)
rs_id = main_recset_id
df = dataframes[rs_id]

# Choose a likely numeric field by column inspection, e.g., 'coverage_percent' or similar
numeric_candidates = [col for col in df.columns if df[col].dtype in ['int64', 'float64']]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Try string columns that can be converted to float
    str_numeric = [col for col in df.columns if df[col].dtype == 'object']
    for col in str_numeric:
        try:
            df[col] = pd.to_numeric(df[col])
            numeric_field = col
            break
        except:
            continue
    else:
        numeric_field = None

if numeric_field is None:
    print("No numeric field found for processing.")
else:
    threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 0
    filtered_df = df[df[numeric_field] > threshold]
    print(f"Filtered rows with {numeric_field} > {threshold:.3f}:")
    print(filtered_df.head())

    norm_col = f"{numeric_field}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized column '{numeric_field}' for filtered records:")
    print(filtered_df[[numeric_field, norm_col]].head())

    # Try to group by a possible categorical field
    group_candidates = [col for col in df.columns if df[col].dtype == 'object' and col != numeric_field]
    group_field = group_candidates[0] if group_candidates else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"\nGrouped by '{group_field}':")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields using `matplotlib`/`seaborn`. All field access by column names reflecting their `@id`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize histogram and boxplot for filtered numeric field if found
if numeric_field:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(filtered_df[numeric_field], ax=axes[0], kde=True, color='salmon')
    axes[0].set_title(f'Distribution of {numeric_field}')
    sns.boxplot(x=filtered_df[numeric_field], ax=axes[1], color='skyblue')
    axes[1].set_title(f'Boxplot of {numeric_field}')
    plt.tight_layout()
    plt.show()

    if group_field:
        plt.figure(figsize=(10,4))
        sns.barplot(x=group_field, y=numeric_field, data=filtered_df)
        plt.xticks(rotation=45)
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
We demonstrated loading, overview, processing, and simple visualization for the FAIR^2 mass spectrometry dataset using the `mlcroissant` library. All references are canonical via entity `@id`. This analysis can be extended for deeper statistical investigation, visualization, or pipeline integration with ML tasks.